In [1]:
!pip install nltk spacy pandas
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 10.7 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 13.9 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 16.6 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 17.2 MB/s eta 0:00:01
     ------------------------------------- - 12.3/12.8 MB 14.0 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 13.9 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import nltk
import spacy
import pandas as pd
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag, RegexpParser

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [3]:
paragraph = """
Natural Language Processing is a rapidly growing field of artificial intelligence.
The intelligent system analyzes customer queries and provides automated responses.
Advanced machine learning algorithms improve the overall user experience.
"""

In [4]:
sentences = sent_tokenize(paragraph)
print("Sentences:")
for s in sentences:
    print("-", s)

Sentences:
- 
Natural Language Processing is a rapidly growing field of artificial intelligence.
- The intelligent system analyzes customer queries and provides automated responses.
- Advanced machine learning algorithms improve the overall user experience.


In [5]:
grammar = r"""
NP: {<DT>?<JJ.*>*<NN.*>+}
"""

chunk_parser = RegexpParser(grammar)

nltk_nps = []

for sentence in sentences:
    words = word_tokenize(sentence)
    pos_tags = pos_tag(words)
    tree = chunk_parser.parse(pos_tags)
    
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
        phrase = " ".join(word for word, tag in subtree)
        nltk_nps.append(phrase)

print("NLTK Noun Phrases:")
for np in nltk_nps:
    print("-", np)

NLTK Noun Phrases:
- Natural Language Processing
- field
- artificial intelligence
- The intelligent system
- customer queries
- automated responses
- Advanced machine
- the overall user experience


In [6]:
doc = nlp(paragraph)

spacy_nps = [chunk.text for chunk in doc.noun_chunks]

print("\nSpaCy Noun Phrases:")
for np in spacy_nps:
    print("-", np)


SpaCy Noun Phrases:
- 
Natural Language Processing
- a rapidly growing field
- artificial intelligence
- The intelligent system
- customer queries
- automated responses
- Advanced machine learning algorithms
- the overall user experience


In [7]:
dependency_nps = []

for token in doc:
    if token.pos_ == "NOUN":
        phrase = " ".join([child.text for child in token.subtree])
        dependency_nps.append(phrase)

print("\nDependency-based Noun Phrases:")
for np in dependency_nps:
    print("-", np)


Dependency-based Noun Phrases:
- 
 Natural Language Processing
- a rapidly growing field of artificial intelligence
- artificial intelligence
- The intelligent system
- customer
- customer queries
- automated responses
- machine
- machine learning
- Advanced machine learning algorithms
- user
- the overall user experience


In [8]:
all_nps = nltk_nps + spacy_nps + dependency_nps

# Normalize
all_nps = [np.lower().strip() for np in all_nps if len(np.split()) > 1]

unique_nps = list(set(all_nps))

print("\nUnique Noun Phrases:")
for np in unique_nps:
    print("-", np)


Unique Noun Phrases:
- a rapidly growing field
- artificial intelligence
- automated responses
- advanced machine
- the intelligent system
- natural language processing
- a rapidly growing field of artificial intelligence
- advanced machine learning algorithms
- customer queries
- machine learning
- the overall user experience


In [9]:
np_counts = Counter(all_nps)

df = pd.DataFrame(np_counts.items(), columns=["Noun Phrase", "Frequency"])
df = df.sort_values(by="Frequency", ascending=False)

df.head(10)

,Noun Phrase,Frequency
0,natural language processing,3
1,artificial intelligence,3
2,the intelligent system,3
3,customer queries,3
4,automated responses,3
6,the overall user experience,3
8,advanced machine learning algorithms,2
5,advanced machine,1
7,a rapidly growing field,1
9,a rapidly growing field of artificial intellig...,1


In [10]:
important_nps = df[df["Frequency"] >= 2]

print("\nImportant Noun Phrases (frequency >= 2):")
print(important_nps)


Important Noun Phrases (frequency >= 2):
                            Noun Phrase  Frequency
0           natural language processing          3
1               artificial intelligence          3
2                the intelligent system          3
3                      customer queries          3
4                   automated responses          3
6           the overall user experience          3
8  advanced machine learning algorithms          2
